In [3]:
pwd

'/Users/nesh/Documents/Repositories/icecontinuum/Sample code/Figure 7. N_limit dependence on c_r and omega_kin '

### Parameters for the bilinear form for $N_{steps}$
Below, coming up with "m" and "b" for

$$
N_{steps}=m \times \omega_{kin} \times c_r + b \ \ \ (1)
$$

where we're using $c_r$ in % (which in this code is called "center_reduction". The above is possible if, in writing the following, assuming (as we did in the code above) a fixed $\omega_{kin}=\omega_{kin}^{ref}$:

$$
N_{steps}=m_1 \times c_r + b_1 \ \ \ (2)
$$

and similarly assuming a fixed $c_r=c_r^{ref}$

$$
N_{steps}=m_2 \times \omega_{kin} + b_2 \ \ \ (3)
$$

If we find that $b_1 \approx b_2$, and therefore it would have to be the case that

$$
{m_1 \over \omega_{kin}^{ref}} = {m_2 \over c_r^{ref}} \ \ \ (4)
$$

Because there are two ways to find $m$ from this, we'll take the averge:

$$
m = {m_1+m_2 \over 2}
$$

$$
b = {b_1+b_2 \over 2}
$$

We're making pint variables below, just to double-check the units.

In [ ]:
# Looping over a range of nu kinetic values
tlast = tlast_msec * 1000 *1.5
tkeep_1Darr = np.linspace(0,tlast,ntimes)
print('dt =', tkeep_1Darr[1]-tkeep_1Darr[0])
sigmaI_ref = sigmaI

nruns = 25
nu_kin_mlyperus_range = np.linspace(nu_kin_mlyperus_ref*0.5,nu_kin_mlyperus_ref*2,nruns)
nsteps_for_nu_kin_range = np.empty(0)

# Loop over values of nu_kin
for this_nu_kin_mlyperus in nu_kin_mlyperus_range:

    # Bundle parameters for ODE solver
    scalar_params = np.array([Nbar, Nstar, sigma0, this_nu_kin_mlyperus, Doverdeltax2, tau_eq])
    
    # Initialize as a pre-equilibrated layer of liquid over ice
    Ntot_init_1D = np.ones(nx)
    NQLL_init_1D = QLC.getNQLL(Ntot_init_1D,Nstar,Nbar)
    
    # Solve
    Ntotkeep_1D, NQLLkeep_1D = QLC.run_f1d(NQLL_init_1D,Ntot_init_1D,tkeep_1Darr,scalar_params,sigmaI_ref,odemethod)
    Nicekeep_1D = Ntotkeep_1D-NQLLkeep_1D

    # Number of steps
    nsteps_run = np.max(Ntotkeep_1D,axis=1) - np.min(Ntotkeep_1D,axis=1)
    nsteps = nsteps_run[-1]
    nsteps_for_nu_kin_range = np.append(nsteps_for_nu_kin_range, nsteps); print(nsteps_for_nu_kin_range)

    # Plot number of steps over time
    nsteps_run = np.max(Ntotkeep_1D,axis=1) - np.min(Ntotkeep_1D,axis=1)
    plt.figure()
    plt.plot(tkeep_1Darr/1e3,nsteps_run,lw=linewidth)
    plt.xlabel(r't ($m s$)',fontsize=fontsize)
    plt.ylabel('# of steps',fontsize=fontsize)
    plt.grid(True)
    plt.title(r'For $\nu_{kin}=$'+str(np.round(this_nu_kin_mlyperus,4)))

In [ ]:
# Analysis of the loop over nu_kin
nu_kin_pint = QLC.get_nu_kin(Temperature,AssignQuantity); print('nu_kin_pint',nu_kin_pint)
tau_eq_pint = AssignQuantity(tau_eq,'microseconds'); print('tau_eq_pint',tau_eq_pint)
nu_kin_mlyperus_range_pint = AssignQuantity(nu_kin_mlyperus_range,'1/microsecond'); print('nu_kin_mlyperus_range_pint',nu_kin_mlyperus_range_pint)
omega_kin_range_pint = tau_eq_pint * nu_kin_mlyperus_range_pint; # print('omega_kin_range_pint',omega_kin_range_pint)
nsteps_for_nu_kin_range_pint = AssignQuantity(nsteps_for_nu_kin_range,'dimensionless')

p = np.polyfit(omega_kin_range_pint.magnitude,nsteps_for_nu_kin_range,1); print(p)
p_omega_kin = p
nsteps_for_nu_kin_range_fit = np.polyval(p,nu_kin_mlyperus_range)
dNsteps_domega_kin = AssignQuantity(p[0],nsteps_for_nu_kin_range_pint.units/omega_kin_range_pint.units); print('dNsteps_domega_kin',dNsteps_domega_kin)

plt.figure()
plt.plot(nu_kin_mlyperus_range,nsteps_for_nu_kin_range,'o')
plt.plot(nu_kin_mlyperus_range,nsteps_for_nu_kin_range_fit)
plt.xlabel(r'$\omega_{kin}$',fontsize=fontsize)
plt.ylabel(r'$N_{steps}$',fontsize=fontsize)
plt.grid(True)

In [ ]:
# Looping over a range of center reduction values
tlast = tlast_msec * 1000 *1.5
tkeep_1Darr = np.linspace(0,tlast,ntimes)
print('dt =', tkeep_1Darr[1]-tkeep_1Darr[0])
center_reduction_ref = center_reduction

nruns = 25
center_reduction_range = np.linspace(center_reduction_ref*.5,center_reduction_ref*1.5,nruns)
nsteps_for_center_reduction_range = np.empty(0)

# Bundle parameters for ODE solver
scalar_params = np.array([Nbar, Nstar, sigma0, nu_kin_mlyperus_ref, Doverdeltax2, tau_eq])

# Loop over values of center_reduction
for this_center_reduction in center_reduction_range:
    
    # Initialize as a pre-equilibrated layer of liquid over ice
    Ntot_init_1D = np.ones(nx)
    NQLL_init_1D = QLC.getNQLL(Ntot_init_1D,Nstar,Nbar)

    # Re-set the imposed supersaturation according to this center reduction
    this_sigmaI = QLC.getsigmaI(x,xmax,this_center_reduction,sigmaIcorner,method='parabolic')
    
    # Solve
    Ntotkeep_1D, NQLLkeep_1D = QLC.run_f1d(NQLL_init_1D,Ntot_init_1D,tkeep_1Darr,scalar_params,this_sigmaI,odemethod)
    Nicekeep_1D = Ntotkeep_1D-NQLLkeep_1D

    # Number of steps
    nsteps_run = np.max(Ntotkeep_1D,axis=1) - np.min(Ntotkeep_1D,axis=1)
    nsteps = nsteps_run[-1]
    nsteps_for_center_reduction_range = np.append(nsteps_for_center_reduction_range, nsteps); print(nsteps_for_center_reduction_range)

    # Plot number of steps over time
    nsteps_run = np.max(Ntotkeep_1D,axis=1) - np.min(Ntotkeep_1D,axis=1)
    plt.figure()
    plt.plot(tkeep_1Darr/1e3,nsteps_run,lw=linewidth)
    plt.xlabel(r't ($m s$)',fontsize=fontsize)
    plt.ylabel('# of steps',fontsize=fontsize)
    plt.grid(True)
    plt.title(r'For $\nu_{kin}=$'+str(np.round(this_nu_kin_mlyperus,4)))

In [ ]:
# Analysis of the loop over center_reduction

center_reduction_range_pint = AssignQuantity(center_reduction_range,'percent')
nsteps_for_center_reduction_range_pint = AssignQuantity(nsteps_for_center_reduction_range,'dimensionless')

p = np.polyfit(center_reduction_range_pint.magnitude,nsteps_for_center_reduction_range,1); print(p)
p_center_reduction = p
nsteps_for_center_reduction_range_fit = np.polyval(p,center_reduction_range)
dNsteps_dcenter_reduction = AssignQuantity(p[0],nsteps_for_center_reduction_range_pint.units/center_reduction_range_pint.units); print('dNsteps_dcenter_reduction',dNsteps_dcenter_reduction)

plt.figure()
plt.plot(center_reduction_range,nsteps_for_center_reduction_range,'o')
plt.plot(center_reduction_range,nsteps_for_center_reduction_range_fit)
plt.xlabel(r'$c_r$ (%)',fontsize=fontsize)
plt.ylabel(r'$N_{steps}$',fontsize=fontsize)
plt.grid(True)

In [ ]:
# These assignments are to correspond to the notes above
m1 = dNsteps_dcenter_reduction; print('m1', m1)
b1 = p_center_reduction[1]; print('b1', b1)
m2 = dNsteps_domega_kin; print('m2', m2)
b2 = p_omega_kin[1]; print('b2', b2)

# Converting to pint form
nu_kin_mlyperus_ref_pint = AssignQuantity(nu_kin_mlyperus_ref,'1/microsecond'); print('nu_kin_mlyperus_ref_pint',nu_kin_mlyperus_ref_pint)
omega_kin_ref_pint = nu_kin_mlyperus_ref_pint * tau_eq_pint; print('omega_kin_ref_pint',omega_kin_ref_pint)
center_reduction_ref_pint = AssignQuantity(center_reduction_ref,'percent'); print('center_reduction_ref_pint',center_reduction_ref_pint)
nsteps_for_omega_kin_range_pint = nsteps_for_nu_kin_range_pint

# Checking the equivalence of Eq. 4 above
LHS = m1/omega_kin_ref_pint; print('LHS', LHS)
RHS = m2/center_reduction_ref_pint; print('RHS', RHS)

# Parameters for the bilinear form
m_combined = (LHS+RHS)/2; print('m_combined',m_combined)
b_combined = (b1+b2)/2; print('b_combined', b_combined)

# Theoretical values predicted by the bilinear parameters m and b
nsteps_for_center_reduction_range_bilinear_fit = m_combined*omega_kin_ref_pint   *center_reduction_range_pint +b_combined
nsteps_for_omega_kin_range_bilinear_fit =        m_combined*omega_kin_range_pint *center_reduction_ref_pint   +b_combined

In [ ]:
# Constructing & saving these graphs as a combined figure
fig = plt.figure(figsize=(10,5))
plt.subplot(1,2,1)
plt.plot(center_reduction_range_pint.magnitude,nsteps_for_center_reduction_range_pint.magnitude,'o')
plt.plot(center_reduction_range_pint.magnitude,nsteps_for_center_reduction_range_bilinear_fit.magnitude)
plt.xlabel(r'$c_r$ (%)',fontsize=fontsize)
plt.ylabel(r'$N_{steps}^{limit}$',fontsize=fontsize)
plt.title('(a)',fontsize=fontsize)
plt.grid(True)

plt.subplot(1,2,2)
plt.plot(omega_kin_range_pint.magnitude,nsteps_for_omega_kin_range_pint.magnitude,'o')
plt.plot(omega_kin_range_pint.magnitude,nsteps_for_omega_kin_range_bilinear_fit.magnitude)
plt.xlabel(r'$\omega_{kin}$',fontsize=fontsize)
# plt.ylabel(r'$N_{steps}^{limit}$',fontsize=fontsize)
plt.title('(b)',fontsize=fontsize)
plt.grid(True)

figurename_Nsteps = 'Figure - Nsteps as a function of omega_kin and c_r.png'
fig.savefig(figurename_Nsteps, dpi=200)